<a href="https://colab.research.google.com/github/paulpdelancy-spec/gdp-dashboard/blob/main/Master__Copy_April_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =================================================================
# 🛡️ YESHUA AI: ENTERPRISE STEWARDSHIP ENGINE (v3.5 - CONSOLIDATED)
# =================================================================

# SECTION 0: INSTALLATION & IMPORTS
!pip install -q -U google-cloud-aiplatform fpdf tqdm beautifulsoup4

import os, datetime, time, pytz, smtplib, requests, random
from fpdf import FPDF
from tqdm import tqdm
from bs4 import BeautifulSoup
from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# SECTION 1: CORE AUTH & CONFIGURATION
auth.authenticate_user()
GEMINI_API_KEY  = userdata.get('GEMINI_API_KEY')
SENDER_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
SENDER_EMAIL    = "paul.pdelancy@gmail.com"
CONTACT_EMAILS  = ["paulacumberbatch@yahoo.com", "paul.pdelancy@gmail.com"]
DRIVE_FOLDER_ID = "1wKOXcL5T2KW4JeOkydB8AxOUuwo1auYI"
OK_TZ = pytz.timezone('America/Chicago')

# SECTION 2: THE BILINGUAL ANALYST
def analyze_batch(batch_data):
    """Processes clusters with a mandatory Sabbath delay to prevent Gemini 429s."""
    url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={GEMINI_API_KEY}"

    prompt = (
        "Role: Expert Stewardship Analyst. "
        "Extract 3-4 high-impact news items for each site. "
        "MANDATORY FORMAT: Provide each point in English, followed immediately by its translation in the site's local language. "
        "If it is a finance site, prioritize market volume and integrity risks."
        f"\n\nDATA:\n{batch_data}"
    )

    try:
        # MANDATORY DELAY: Protects the Gemini API quota
        time.sleep(40)
        response = requests.post(url, json={"contents": [{"parts": [{"text": prompt}]}]}, timeout=45)
        if response.status_code == 200:
            return response.json()["candidates"][0]["content"]["parts"][0]["text"]
        else:
            print(f"⚠️ AI Server status {response.status_code}. Cooling down 60s...")
            time.sleep(60)
            return "⚠️ Segment stalled. Manual review required."
    except Exception as e:
        return f"⚠️ Intelligence Engine Timeout: {str(e)}"

# SECTION 3: THE PDF GENERATOR
def create_stewardship_pdf(summary_text):
    print("📄 Creating Adobe PDF Report...")
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(200, 10, txt="Stewardship Pulse: Global Intelligence", ln=True, align='C')
    pdf.set_font("Arial", size=11)
    pdf.ln(10)
    clean_text = summary_text.encode('latin-1', 'ignore').decode('latin-1')
    pdf.multi_cell(0, 10, txt=clean_text)
    pdf_name = f"Stewardship_Pulse_{datetime.datetime.now().strftime('%H%M')}.pdf"
    pdf.output(pdf_name)
    return pdf_name

# SECTION 4: DRIVE SYNC
def sync_and_purge(file_path):
    if not file_path or not os.path.exists(file_path): return "Link Unavailable"
    try:
        service = build('drive', 'v3')
        meta = {'name': os.path.basename(file_path), 'parents': [DRIVE_FOLDER_ID]}
        media = MediaFileUpload(file_path, mimetype='application/pdf')
        file = service.files().create(body=meta, media_body=media, fields='id, webViewLink').execute()
        service.permissions().create(fileId=file.get('id'), body={'type': 'anyone', 'role': 'reader'}).execute()
        os.remove(file_path)
        return file.get('webViewLink')
    except Exception as e:
        print(f"⚠️ Drive Sync Error: {e}")
        return "Link Unavailable"

# SECTION 5: EMAIL DELIVERY
def deliver_email(html_content):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = "🛡️ Stewardship Pulse: Global Bilingual Audit"
    msg["From"], msg["To"] = SENDER_EMAIL, ", ".join(CONTACT_EMAILS)
    msg.attach(MIMEText(html_content, "html"))
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
            s.login(SENDER_EMAIL, SENDER_PASSWORD)
            s.sendmail(SENDER_EMAIL, CONTACT_EMAILS, msg.as_string())
        print("✅ Delivery Confirmed.")
    except: print("❌ Email Delivery Failed.")

# SECTION 6: THE 72-SITE REGISTRY

# SECTION 6: THE FULL 72-SITE REGISTRY
SPORTS_SITES = [
    {"url": "https://www.afa.com.ar/es/", "name": "AFA Argentina"},
    {"url": "https://dimayor.com.co/", "name": "Dimayor Colombia"},
    {"url": "https://www.cfl.ca/", "name": "CFL Canada"},
    {"url": "https://www.mlb.com/", "name": "MLB USA"},
    {"url": "https://www.mlssoccer.com/", "name": "MLS USA"},
    {"url": "https://www.nba.com/", "name": "NBA USA"},
    {"url": "https://www.nfl.com/", "name": "NFL USA"},
    {"url": "https://www.nhl.com/", "name": "NHL USA/Canada"},
    {"url": "https://www.ligamx.net/", "name": "Liga MX Mexico"},
    {"url": "https://www.cbf.com.br/", "name": "CBF Brazil"},
    {"url": "https://www.premierleague.com/", "name": "Premier League UK"},
    {"url": "https://www.bundesliga.com/en/bundesliga/", "name": "Bundesliga Germany"},
    {"url": "https://www.bundesliga.at/de", "name": "Bundesliga Austria"},
    {"url": "https://www.laliga.com/en-GB", "name": "La Liga Spain"},
    {"url": "https://www.legaseriea.it/it", "name": "Serie A Italy"},
    {"url": "https://ligue1.com/", "name": "Ligue 1 France"},
    {"url": "https://top14.lnr.fr/", "name": "Top 14 Rugby France"},
    {"url": "https://www.efl.com/", "name": "EFL England"},
    {"url": "https://eredivisie.nl/", "name": "Eredivisie Netherlands"},
    {"url": "https://www.ligaportugal.pt/", "name": "Liga Portugal"},
    {"url": "https://sfl.ch/", "name": "Swiss Football League"},
    {"url": "https://www.tff.org/", "name": "TFF Turkey"},
    {"url": "https://www.afl.com.au/", "name": "AFL Australia"},
    {"url": "https://www.nrl.com/", "name": "NRL Australia"},
    {"url": "http://eng.koreabaseball.com/", "name": "KBO Korea"},
    {"url": "https://www.jleague.jp/en/", "name": "J-League Japan"},
    {"url": "https://www.iplt20.com/", "name": "IPL India"},
    {"url": "https://www.indiansuperleague.com/", "name": "ISL India"},
    {"url": "https://sports.sina.com.cn/csl/", "name": "CSL China"}
]

FINANCE_SITES = [
    {"url": "https://www.nyse.com/index", "name": "NYSE USA"},
    {"url": "https://www.nasdaq.com/", "name": "NASDAQ USA"},
    {"url": "https://www.cmegroup.com/", "name": "CME Group USA"},
    {"url": "https://www.cboe.com/", "name": "CBOE USA"},
    {"url": "https://www.nadex.com/", "name": "NADEX USA"},
    {"url": "https://www.miaxglobal.com/", "name": "MIAX USA"},
    {"url": "https://www.tmx.com/", "name": "TMX Canada"},
    {"url": "https://www.m-x.ca/en/", "name": "MX Canada"},
    {"url": "https://www.euronext.com/en", "name": "Euronext Europe"},
    {"url": "https://www.eurex.com/ex-en/", "name": "Eurex Germany"},
    {"url": "https://deutsche-boerse.com/dbg-en/", "name": "Deutsche Boerse"},
    {"url": "https://www.lseg.com/en", "name": "LSEG London"},
    {"url": "https://www.lme.com/", "name": "LME London"},
    {"url": "https://www.six-group.com/en/", "name": "SIX Swiss Exchange"},
    {"url": "https://www.nordpoolgroup.com/", "name": "Nord Pool Energy"},
    {"url": "https://www.epexspot.com/en", "name": "EPEX Spot Energy"},
    {"url": "https://www.bolsasymercados.es/ing/Home", "name": "BME Spain"},
    {"url": "https://www.dgcx.ae/", "name": "DGCX Dubai"},
    {"url": "https://www.gulfmerc.com/", "name": "Gulf Mercantile"},
    {"url": "https://www.nasdaqdubai.com/", "name": "NASDAQ Dubai"},
    {"url": "https://www.jse.co.za/trade", "name": "JSE South Africa"},
    {"url": "https://www.ea-africaexchange.com/", "name": "EA Africa Exchange"},
    {"url": "https://www.ecx.com.et/", "name": "ECX Ethiopia"},
    {"url": "https://www.hkex.com.hk/", "name": "HKEX Hong Kong"},
    {"url": "https://www.jpx.co.jp/english/", "name": "JPX Japan"},
    {"url": "https://www.tfx.co.jp/en/", "name": "TFX Japan"},
    {"url": "https://www.sse.com.cn/", "name": "SSE Shanghai"},
    {"url": "https://www.szse.cn/English/", "name": "SZSE Shenzhen"},
    {"url": "http://www.shfe.com.cn/en/", "name": "SHFE Shanghai"},
    {"url": "http://www.czce.com.cn/en/", "name": "CZCE Zhengzhou"},
    {"url": "http://www.dce.com.cn/en/", "name": "DCE Dalian"},
    {"url": "https://www.bseindia.com/", "name": "BSE India"},
    {"url": "https://www.nseindia.com/", "name": "NSE India"},
    {"url": "https://www.mcxindia.com/", "name": "MCX India"},
    {"url": "https://www.ncdex.com/markets/", "name": "NCDEX India"},
    {"url": "https://global.krx.co.kr/", "name": "KRX Korea"},
    {"url": "https://www.sgx.com/", "name": "SGX Singapore"},
    {"url": "https://www.twse.com.tw/en/", "name": "TWSE Taiwan"},
    {"url": "https://www.idx.co.id/en-us", "name": "IDX Indonesia"},
    {"url": "https://www.bursamalaysia.com/", "name": "Bursa Malaysia"},
    {"url": "https://www.set.or.th/en/home", "name": "SET Thailand"},
    {"url": "https://www.pse.com.ph/", "name": "PSE Philippines"},
    {"url": "https://mce.mn/", "name": "MCE Mongolia"}
]


# SECTION 7: THE PERSISTENT SESSION PIPELINE (v3.5)
def execute_pulse_cycle():
    print(f"\n🔄 PERSISTENT CYCLE START — {datetime.datetime.now(OK_TZ)}")
    master_report = "# 🛡️ STEWARDSHIP INTELLIGENCE: GLOBAL BILINGUAL AUDIT\n\n"
    all_sites = SPORTS_SITES + FINANCE_SITES
    random.shuffle(all_sites)

    session = requests.Session()
    organic_sources = ["https://www.google.com/", "https://www.bing.com/", "https://www.yahoo.com/"]

    current_batch_text = ""
    sites_in_current_batch = 0
    target_batch_size = 3

    for i, site in enumerate(tqdm(all_sites, desc="🔎 Harvesting")):
        try:
            # EXTENDED JITTER: Mimics human reading time
            time.sleep(random.uniform(12.0, 18.0))





            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
                "Accept": "text/html,application/xhtml+xml,xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
                "Accept-Encoding": "gzip, deflate, br", # Added for modern compression
                "Accept-Language": "en-US,en;q=0.9",
                "Sec-Ch-Ua": '"Google Chrome";v="123", "Not:A-Brand";v="8", "Chromium";v="123"', # Client Hints
                "Sec-Ch-Ua-Mobile": "?0",
                "Sec-Ch-Ua-Platform": '"Windows"',
                "Referer": "https://www.google.com/",
                "DNT": "1", # Do Not Track header mimics privacy-conscious humans
                "Connection": "keep-alive"
            }



            res = session.get(site["url"], headers=headers, timeout=25, allow_redirects=True)

            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")
                # Strip clutter
                for s in soup(["script", "style", "nav", "footer", "header", "aside", "form", "button"]): s.decompose()
                clean_text = soup.get_text(separator=' ')[:2500]
                current_batch_text += f"\n--- SOURCE: {site['name']} ---\n{clean_text}\n"
                sites_in_current_batch += 1
            else:
                master_report += f"## 📍 {site['name']}\n• Barrier Encountered: Status {res.status_code}\n\n"

            if sites_in_current_batch >= target_batch_size or i == len(all_sites) - 1:
                print(f"\n🧠 Processing Batch Intelligence...")
                batch_result = analyze_batch(current_batch_text)
                master_report += f"{batch_result}\n\n"
                current_batch_text = ""
                sites_in_current_batch = 0

        except Exception as e:
            master_report += f"## 📍 {site['name']}\n• Connection Reset\n\n"

    pdf_link = sync_and_purge(create_stewardship_pdf(master_report))
    deliver_email(f"<h2>🛡️ Stewardship Pulse</h2><p><a href='{pdf_link}'>📄 DOWNLOAD PDF</a></p>")
    print(f"📧 Audit Delivered Successfully at {datetime.datetime.now(OK_TZ)}")

# SECTION 8: INDEFINITE LOOP
if __name__ == "__main__":
    while True:
        try:
            execute_pulse_cycle()
            print("💤 Resting 4 hours...")
            time.sleep(14400)
        except KeyboardInterrupt:
            print("🛑 Manual Stop Detected.")
            break
        except Exception as e:
            print(f"⚠️ Loop Error: {e}. Restarting in 60s...")
            time.sleep(60)



# SECTION 9: STEWARDSHIP VOICE SYNTHESIS (New)
from google.cloud import texttospeech

def generate_stewardship_voice(summary_text):
    print("🎙️ Synthesizing Stewardship Voice Briefing...")
    client = texttospeech.TextToSpeechClient()

    # We strip the Markdown characters (#, *, ---) so the AI reads it cleanly
    clean_speech = summary_text.replace("#", "").replace("*", "").replace("---", "Next point.")

    input_text = texttospeech.SynthesisInput(text=clean_speech[:4000]) # Voice limit check

    # Selection: 'en-US-Neural2-F' is a professional, calm, authoritative voice
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="en-US-Neural2-F"
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    response = client.synthesize_speech(
        input=input_text, voice=voice, audio_config=audio_config
    )

    mp3_name = f"Stewardship_Voice_{datetime.datetime.now().strftime('%H%M')}.mp3"
    with open(mp3_name, "wb") as out:
        out.write(response.audio_content)

    return mp3_name







     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.7 MB/s eta 0:00:00

🔄 PERSISTENT CYCLE START — 2026-04-02 07:25:25.910838-05:00


🔎 Harvesting:   4%|▍         | 3/72 [00:45<17:23, 15.12s/it]


🧠 Processing Batch Intelligence...


🔎 Harvesting:  10%|▉         | 7/72 [02:48<23:48, 21.98s/it]


🧠 Processing Batch Intelligence...


...


🧠 Processing Batch Intelligence...
⚠️ AI Server status 429. Cooling down 60s...


🔎 Harvesting:  74%|███████▎  | 53/72 [31:58<08:45, 27.63s/it]


🧠 Processing Batch Intelligence...
⚠️ AI Server status 429. Cooling down 60s...


🔎 Harvesting:  79%|███████▉  | 57/72 [34:37<07:01, 28.13s/it]


🧠 Processing Batch Intelligence...
⚠️ AI Server status 429. Cooling down 60s...
